In [ ]:
import sqlite3
import pandas as pd

# 1. Connect to the shared database
# conn = sqlite3.connect("../../data/processed/cryptocurrency-110826.db")
conn = sqlite3.connect("../../data/processed/reddit-data-180826.db")

# 2. Extract exactly what your model needs
query_cryptocurrency = """
SELECT id, subreddit, timestamp, title, description, (title || ' ' || COALESCE(description, '')) AS full_text
FROM posts
WHERE (subreddit IS 'cryptocurrency')
"""
df_cryptocurrency = pd.read_sql(query_cryptocurrency, conn)

query_bitcoin = """
SELECT id, subreddit, timestamp, title, description, (title || ' ' || COALESCE(description, '')) AS full_text
FROM posts
WHERE (subreddit IS 'bitcoin')
"""
df_bitcoin = pd.read_sql(query_bitcoin, conn)

query_cryptomarkets = """
SELECT id, subreddit, timestamp, title, description, (title || ' ' || COALESCE(description, '')) AS full_text
FROM posts
WHERE (subreddit IS 'cryptomarkets')
"""
df_cryptomarkets = pd.read_sql(query_cryptomarkets, conn)

# 3. Pass it to your verification model
# Example: 
# predictions = my_claim_verifier_model.predict(df_claims['full_text'].tolist())

# Display the first few rows to verify it worked
df_cryptocurrency.head()
df_cryptocurrency["TITLE"]

In [ ]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd

# Add src to path
sys.path.append(str(Path.cwd().parent.parent / "src"))
from verifier.claim_verifier import ClaimVerifier

# 1. Connect to SQLite and fetch sample posts
conn = sqlite3.connect("../../data/processed/reddit-data-180826.db")
posts = pd.read_sql("""
    SELECT id, subreddit, timestamp, title, description, (title || ' ' || COALESCE(description, '')) AS full_text
    FROM posts
    WHERE (subreddit IS 'cryptocurrency')
""", conn)

# 2. Instantiate Verifier
verifier = ClaimVerifier()

# 3. Test verification on post titles
results = []
for _, row in posts.iterrows():
    claim = row["full_text"]
    verdict = verifier.verify_claim(claim)
    results.append({
        "post_id": row["ID"],
        "claim": claim,
        "veracity": verdict["veracity"],
        "confidence": verdict["confidence"],
        "source": verdict["source"],
        "evidence": verdict["evidence"]
    })

# 4. Display results
verification_df = pd.DataFrame(results)
display(verification_df)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# 1. Connect to your database
client = chromadb.PersistentClient(path="../../data/processed/chroma_db")
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
coll = client.get_collection("crypto_factual_evidence", embedding_function=emb_fn)

# --- QUERY 1: Exact Metadata Filter (like SELECT * WHERE source = 'CoinDesk') ---
coindesk_docs = coll.get(
    where={"source": "CoinDesk"},
    limit=5
)
for doc, meta in zip(coindesk_docs["documents"], coindesk_docs["metadatas"]):
    print(f"[{meta['published_at']}] {doc[:100]}... (URL: {meta.get('url')})")

# --- QUERY 2: Keyword Search (like SELECT * WHERE document LIKE '%Bitcoin%') ---
btc_docs = coll.get(
    where_document={"$contains": "Bitcoin"},
    limit=5
)

# --- QUERY 3: Hybrid Search (Semantic query + Metadata constraint) ---
# Finds news specifically from Decrypt about Ethereum staking
results = coll.query(
    query_texts=["Ethereum staking withdrawals"],
    where={"source": "Decrypt"},
    n_results=3
)

In [ ]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd

# 1. Add src to Python path
sys.path.append(str(Path.cwd().parent.parent / "src"))
from verifier.decomposer import decompose_post
from verifier.claim_verifier import ClaimVerifier

# 2. Query posts from SQLite
conn = sqlite3.connect("../../data/processed/reddit-data-180826.db")
posts_df = pd.read_sql("""
    SELECT id, subreddit, timestamp, title, description, (title || ' ' || COALESCE(description, '')) AS full_text
    FROM posts
    WHERE (subreddit IS 'cryptocurrency')
""", conn)

# 3. Decompose all posts into atomic claims
extracted_claims = []
for _, row in posts_df.iterrows():
    post_claims = decompose_post(row["ID"], row["TITLE"], row["DESCRIPTION"])
    extracted_claims.extend(post_claims)

claims_df = pd.DataFrame(extracted_claims)
print(f"Extracted {len(claims_df)} candidate claims from {len(posts_df)} posts.")

# 4. Run NLI Verification against ChromaDB
verifier = ClaimVerifier()
verification_results = []

for _, claim_row in claims_df.iterrows():
    claim = claim_row["claim_text"]
    verdict = verifier.verify_claim(claim)
    
    verification_results.append({
        "post_id": claim_row["post_id"],
        "source": claim_row["source_segment"],
        "extracted_claim": claim,
        "veracity": verdict["veracity"],
        "confidence": verdict["confidence"],
        "evidence_source": verdict["source"],
        "evidence_snippet": verdict["evidence"]
    })

results_df = pd.DataFrame(verification_results)
display(results_df.head(50))

In [ ]:
df_claims.full_text

In [ ]:
len(df_bitcoin)